# GLM and Permutation Tests

In [8]:
from statsmodels.stats.multitest import multipletests

from glm_permutation_tests import run_glm, plot_glm_coefficients, run_permutation_anova
# Notebook header
%load_ext autoreload
%autoreload 2
import pandas as pd
from analyses.spike_count import prepare_binned_spike_data, aggregate_trial_level
import warnings
from scipy.stats import ConstantInputWarning

warnings.simplefilter("ignore", ConstantInputWarning)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load Session for Analysis

In [9]:
# Step 1: Load and prepare data
date = "2023-09-26"
round_no = 3
bin_size = 0.05

analysis_df = prepare_binned_spike_data(date, round_no, bin_size)


Reading Intan Technologies RHD2000 Data File, Version 3.2

Found 24 amplifier channels.
Found 0 auxiliary input channels.
Found 0 supply voltage channels.
Found 0 board ADC channels.
Found 2 board digital input channels.
Found 0 board digital output channels.
Found 0 temperature sensors channels.

Header file contains no data.  Amplifiers were sampled at 20.00 kS/s.
Done!  Elapsed time: 0.0 seconds


## Effect of Stimulus Identity for Zombies

In [10]:
zombies_df = analysis_df[analysis_df['MonkeyGroup'] == 'Zombies']
formula = "SpikeCount ~ C(MonkeyName)"  # Stimulus identity


## GLM

In [11]:
glm_results = run_glm(zombies_df, formula=formula)
print(glm_results.head())
# ---- multiple comparison correction ---
# select p-value column from glm_results
pvals = glm_results['P>|z|']

Running GLM per neuron: 100%|██████████| 32/32 [00:00<00:00, 63.31it/s]

                   index      Coef.      Std.Err.             z     P>|z|  \
0              Intercept -28.128402  38133.918272 -7.376216e-04  0.999411   
1  C(MonkeyName)[T.143H]  23.364094  38133.918275  6.126854e-04  0.999511   
2  C(MonkeyName)[T.151J]  -0.000007  55501.364509 -1.290195e-10  1.000000   
3   C(MonkeyName)[T.67G]  -0.000007  53962.051818 -1.327011e-10  1.000000   
4   C(MonkeyName)[T.69X]  23.927983  38133.918274  6.274724e-04  0.999499   

          [0.025         0.975]                           NeuronID  
0  -74769.234805   74712.978001  2023-09-26_3_Channel.C_012_Unit 1  
1  -74717.742316   74764.470503  2023-09-26_3_Channel.C_012_Unit 1  
2 -108780.675539  108780.675524  2023-09-26_3_Channel.C_012_Unit 1  
3 -105763.678102  105763.678088  2023-09-26_3_Channel.C_012_Unit 1  
4  -74717.178424   74765.034389  2023-09-26_3_Channel.C_012_Unit 1  


## GLM Multiple Comparison Correction

In [12]:
# ---- multiple comparison correction ---
# select p-value column from glm_results
pvals = glm_results['P>|z|']
# correction method: 'fdr_bh' (False Discovery Rate, Benjamini/Hochberg)
reject, pvals_corrected, _, _ = multipletests(pvals, method='fdr_bh')

# add corrected p-value and reject to glm_results
glm_results['pval_corrected'] = pvals_corrected
glm_results['significant'] = reject

In [13]:
# glm_results.to_excel('glm_results.xlsx')

In [14]:
plot_glm_coefficients(glm_results)

## Permutation ANOVA

In [30]:
zombies_trial_df= aggregate_trial_level(zombies_df)
# unique_neurons = zombies_trial_df['NeuronID'].unique()
# neuron_df = zombies_trial_df[zombies_trial_df['NeuronID'] == unique_neurons[0]]

In [29]:
perm_anova_results = run_permutation_anova(zombies_trial_df, category_col='MonkeyName', plot=False)

Running permutation ANOVA per neuron: 100%|██████████| 32/32 [00:07<00:00,  4.39it/s]


Results:
                             NeuronID  F-statistic  p-value
0   2023-09-26_3_Channel.C_002_Unit 1     5.116338    0.001
1          2023-09-26_3_Channel.C_004     1.186998    0.294
2          2023-09-26_3_Channel.C_006     0.472202    0.905
3   2023-09-26_3_Channel.C_007_Unit 1     0.761809    0.643
4   2023-09-26_3_Channel.C_007_Unit 2     0.990568    0.452
5   2023-09-26_3_Channel.C_009_Unit 1     0.783351    0.618
6   2023-09-26_3_Channel.C_009_Unit 2     1.761453    0.089
7          2023-09-26_3_Channel.C_010     0.818452    0.900
8          2023-09-26_3_Channel.C_011     0.848581    0.581
9   2023-09-26_3_Channel.C_012_Unit 1     1.091445    0.357
10  2023-09-26_3_Channel.C_012_Unit 2     0.946060    0.475
11         2023-09-26_3_Channel.C_013     1.673057    0.113
12  2023-09-26_3_Channel.C_014_Unit 1     0.676258    0.769
13  2023-09-26_3_Channel.C_014_Unit 2     0.361645    0.957
14  2023-09-26_3_Channel.C_017_Unit 1     0.433197    0.917
15  2023-09-26_3_Channel.C_017